# 12_Full Project Outcome Labels

This notebook builds the full-project HbA1c outcome labels for the event cohort created in notebook 10 and prepared for modeling in notebook 11. It pulls follow-up HbA1c support, selects the closest valid follow-up HbA1c in the 3-15 month window, applies the unchanged-therapy approximation, and writes the final outcome label file expected by notebook 11.


## 2. Scope

This notebook is the missing outcome-label step for the full-project workflow. It assumes the following already exist:

- `data/full_project/cohorts/full_project_candidate_cohort.csv`
- `data/full_project/candidate_support/*__all_glucose_lowering_meds.csv`

It produces:

- `data/full_project/outcomes/full_project_outcome_hba1c.csv`

That file is the label file notebook 11 is already prepared to consume.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
PROJECT_READY_DIR = DATA_DIR / 'project_ready'
FULL_PROJECT_DIR = DATA_DIR / 'full_project'
FULL_COHORT_DIR = FULL_PROJECT_DIR / 'cohorts'
FULL_SUPPORT_DIR = FULL_PROJECT_DIR / 'candidate_support'
FULL_OUTCOME_DIR = FULL_PROJECT_DIR / 'outcomes'
FULL_OUTCOME_SUPPORT_DIR = FULL_OUTCOME_DIR / 'followup_hba1c_support'

FULL_OUTCOME_DIR.mkdir(parents=True, exist_ok=True)
FULL_OUTCOME_SUPPORT_DIR.mkdir(parents=True, exist_ok=True)

FULL_CANDIDATE_COHORT_PATH = FULL_COHORT_DIR / 'full_project_candidate_cohort.csv'
FULL_PROJECT_OUTCOME_PATH = FULL_OUTCOME_DIR / 'full_project_outcome_hba1c.csv'
FULL_PROJECT_OUTCOME_ELIGIBLE_PATH = FULL_OUTCOME_DIR / 'full_project_outcome_hba1c_eligible.csv'

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 3. Load The Full Candidate Cohort And Rebuild Event IDs


In [2]:
full_project_candidate_cohort = pd.read_csv(
    FULL_CANDIDATE_COHORT_PATH,
    parse_dates=['index_date', 'first_t2d_date', 'baseline_hba1c_date']
) if FULL_CANDIDATE_COHORT_PATH.exists() else pd.DataFrame()

if full_project_candidate_cohort.empty:
    event_lookup = pd.DataFrame(columns=['event_id', 'patient_id', 'drug_class', 'index_date'])
else:
    event_lookup = (
        full_project_candidate_cohort[['patient_id', 'drug_class', 'index_date']]
        .drop_duplicates()
        .sort_values(['patient_id', 'index_date', 'drug_class'])
        .reset_index(drop=True)
        .copy()
    )
    event_lookup['patient_id'] = event_lookup['patient_id'].astype('string')
    event_lookup['event_id'] = ['event_%07d' % (i + 1) for i in range(len(event_lookup))]

outcome_event_cohort = event_lookup.merge(
    full_project_candidate_cohort,
    on=['patient_id', 'drug_class', 'index_date'],
    how='left',
    validate='one_to_one'
) if not event_lookup.empty else pd.DataFrame()

print('Outcome event rows:', len(outcome_event_cohort))
print('Outcome event patients:', outcome_event_cohort['patient_id'].nunique() if not outcome_event_cohort.empty else 0)
display(outcome_event_cohort.head(10))


Outcome event rows: 120224
Outcome event patients: 94651


,patient_id,drug_class,index_date,event_id,age_at_index,sex,first_t2d_date,baseline_hba1c_date,code_system,code,baseline_hba1c,units_of_measure,days_from_index
0,#A##B,GLP1,2019-07-03,event_0000001,74.0,M,2019-03-28,2019-06-19,LOINC,17856-6,8.0,%,14
1,#A##B,SGLT2,2023-12-14,event_0000002,78.0,M,2019-03-28,2023-12-04,LOINC,17856-6,7.5,%,10
2,#A#0B,GLP1,2023-08-16,event_0000003,53.0,F,2022-03-18,2023-08-12,LOINC,4548-4,7.2,%,4
3,#A#1B,SGLT2,2015-07-02,event_0000004,73.0,M,2014-10-13,2015-07-02,LOINC,4548-4,8.8,%,0
4,#A#2,DPP4,2018-08-02,event_0000005,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,0
5,#A#2,SGLT2,2018-08-16,event_0000006,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,14
6,#A#2,GLP1,2020-02-24,event_0000007,49.0,F,2018-04-24,2020-02-06,LOINC,4548-4,9.1,%,18
7,#A#9,DPP4,2008-03-10,event_0000008,67.0,M,2005-10-05,2007-11-17,LOINC,4548-4,7.3,%,114
8,#A#AB,GLP1,2022-02-09,event_0000009,29.0,F,2021-12-27,2022-02-01,LOINC,4548-4,9.1,%,8
9,#A#AD,GLP1,2025-11-19,event_0000010,66.0,M,2025-06-12,2025-09-26,LOINC,4548-4,8.5,%,54


## 4. Build Follow-Up HbA1c Support Batches


In [3]:
OUTCOME_SUPPORT_BATCH_SIZE = 5000

outcome_support_patients = sorted(outcome_event_cohort['patient_id'].dropna().astype('string').unique().tolist()) if not outcome_event_cohort.empty else []
outcome_support_batches = [
    outcome_support_patients[i:i + OUTCOME_SUPPORT_BATCH_SIZE]
    for i in range(0, len(outcome_support_patients), OUTCOME_SUPPORT_BATCH_SIZE)
]
outcome_support_batch_summary = pd.DataFrame({
    'batch_id': [f'batch_{i+1:03d}' for i in range(len(outcome_support_batches))],
    'n_patients': [len(b) for b in outcome_support_batches],
})

print('Outcome support patients:', len(outcome_support_patients))
print('Outcome support batches:', len(outcome_support_batches))
display(outcome_support_batch_summary.head(20))


Outcome support patients: 94651
Outcome support batches: 19


,batch_id,n_patients
0,batch_001,5000
1,batch_002,5000
2,batch_003,5000
3,batch_004,5000
4,batch_005,5000
5,batch_006,5000
6,batch_007,5000
7,batch_008,5000
8,batch_009,5000
9,batch_010,5000


## 5. Define Full-Project Follow-Up HbA1c Pulls

This uses the exact validated HbA1c code pairs observed in the pilot follow-up support file, but queries the renamed `aa_` source tables.


In [4]:
pilot_followup_hba1c = pd.read_csv(PROJECT_READY_DIR / 'outcome_support' / 'followup_hba1c_support' / 'batch_001__followup_hba1c.csv')
FOLLOWUP_HBA1C_CODE_PAIRS = pilot_followup_hba1c[['code_system', 'code']].dropna().drop_duplicates().values.tolist()


def sql_string_list(values: list[str]) -> str:
    return ', '.join(["'" + str(v).replace("'", "''") + "'" for v in values])


def patient_filter_sql(patient_ids: list[str], column_name: str = 'lr.patient_id') -> str:
    return f"{column_name} IN ({sql_string_list(patient_ids)})"


def build_code_clause(alias: str, code_pairs: list[list[str]]) -> str:
    return ' OR '.join([f"({alias}.code_system = '{cs}' AND {alias}.code = '{code}')" for cs, code in code_pairs])


def build_followup_hba1c_queries(batch: list[str], batch_id: str) -> dict[str, str]:
    pf = patient_filter_sql(batch, 'lr.patient_id')
    hba1c_code_clause = build_code_clause('lr', FOLLOWUP_HBA1C_CODE_PAIRS)
    return {
        f'{batch_id}__followup_hba1c': f'''SELECT
    lr.patient_id,
    lr.date,
    lr.code_system,
    lr.code,
    lr.lab_result_num_val,
    lr.units_of_measure
FROM public.aa_lab_result lr
WHERE {pf}
  AND lr.lab_result_num_val IS NOT NULL
  AND ({hba1c_code_clause})'''
    }

followup_hba1c_query_bundle = {}
for i, batch in enumerate(outcome_support_batches, start=1):
    followup_hba1c_query_bundle.update(build_followup_hba1c_queries(batch, f'batch_{i:03d}'))

followup_hba1c_export_paths = {
    name: FULL_OUTCOME_SUPPORT_DIR / f'{name}.csv'
    for name in followup_hba1c_query_bundle
}

print('Defined full-project follow-up HbA1c queries:', len(followup_hba1c_query_bundle))
display(pd.Series(list(followup_hba1c_query_bundle.keys())[:20], name='query_name').to_frame())


Defined full-project follow-up HbA1c queries: 19


,query_name
0,batch_001__followup_hba1c
1,batch_002__followup_hba1c
2,batch_003__followup_hba1c
3,batch_004__followup_hba1c
4,batch_005__followup_hba1c
5,batch_006__followup_hba1c
6,batch_007__followup_hba1c
7,batch_008__followup_hba1c
8,batch_009__followup_hba1c
9,batch_010__followup_hba1c


## 6. Controlled Full-Project Follow-Up HbA1c Export Runner


In [5]:
db_config = {
  'dbname': 'diabetes_trinetx',
  'user': 'u1523752',
  'host': 'kp196.ipoib.kingspeak.peaks',
  'port': 5432,
  'password': 'u1523752'
}


In [7]:
RUN_FULL_OUTCOME_EXPORTS = True

# Start small, then widen after the first successful outcome batch.
OUTCOME_BATCH_SLICE = slice(1, 19)
SELECTED_OUTCOME_BATCHES = outcome_support_batch_summary['batch_id'].tolist()[OUTCOME_BATCH_SLICE]

selected_outcome_exports = [
    (followup_hba1c_export_paths[name], followup_hba1c_query_bundle[name])
    for name in followup_hba1c_query_bundle
    if name.split('__')[0] in SELECTED_OUTCOME_BATCHES
]

print('Selected outcome batch slice:', OUTCOME_BATCH_SLICE)
print('Selected outcome batches:', SELECTED_OUTCOME_BATCHES)
print('Selected outcome exports:', len(selected_outcome_exports))
display(pd.DataFrame({'path': [str(p) for p, _ in selected_outcome_exports]}).head(20))


def resolve_db_config_local() -> dict:
    cfg = globals().get('db_config', {})
    resolved = {
        'host': cfg.get('host') or cfg.get('HOST'),
        'port': int(cfg.get('port') or cfg.get('PORT') or 5432),
        'dbname': cfg.get('dbname') or cfg.get('DBNAME') or cfg.get('database') or cfg.get('DATABASE'),
        'user': cfg.get('user') or cfg.get('USER'),
        'password': cfg.get('password') or cfg.get('PASSWORD'),
    }
    missing = [k for k, v in resolved.items() if v in (None, '')]
    return {'db_config': resolved, 'missing': missing}

resolution = resolve_db_config_local()
print('DB config status:', {k: ('missing' if k in resolution['missing'] else 'set') for k in ['host', 'port', 'dbname', 'user', 'password']})

if RUN_FULL_OUTCOME_EXPORTS:
    import psycopg2
    if resolution['missing']:
        print('Missing DB config values:', resolution['missing'])
    else:
        db_cfg = resolution['db_config']
        for out_path, query in selected_outcome_exports:
            copy_sql = f"COPY ({query}) TO STDOUT WITH CSV HEADER"
            with psycopg2.connect(**db_cfg) as conn:
                with conn.cursor() as cur, open(out_path, 'w', newline='') as f:
                    cur.copy_expert(copy_sql, f)
            print(out_path)
else:
    print('Set RUN_FULL_OUTCOME_EXPORTS = True when ready.')


Selected outcome batch slice: slice(1, 19, None)
Selected outcome batches: ['batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
Selected outcome exports: 18


,path
0,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
1,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
2,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
3,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
4,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
5,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
6,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
7,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
8,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
9,/uufs/chpc.utah.edu/common/home/kukhareva-grou...


DB config status: {'host': 'set', 'port': 'set', 'dbname': 'set', 'user': 'set', 'password': 'set'}
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/followup_hba1c_support/batch_002__followup_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/followup_hba1c_support/batch_003__followup_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/followup_hba1c_support/batch_004__followup_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/followup_hba1c_support/batch_005__followup_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/followup_hba1c_support/batch_006__followup_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/follo

## 7. Load Completed Follow-Up HbA1c Support And Select The Closest 12-Month Outcome

This section is chunkwise and safe to rerun after more outcome-support batches have been exported.


In [8]:
OUTCOME_SELECTION_CHUNKSIZE = 250_000

followup_hba1c_files = sorted(FULL_OUTCOME_SUPPORT_DIR.glob('batch_*__followup_hba1c.csv'))

followup_support_summary = pd.DataFrame([
    {
        'files': len(followup_hba1c_files),
        'size_mb': round(sum(p.stat().st_size for p in followup_hba1c_files) / (1024 * 1024), 2) if followup_hba1c_files else 0.0,
        'first_file': followup_hba1c_files[0].name if followup_hba1c_files else pd.NA,
        'last_file': followup_hba1c_files[-1].name if followup_hba1c_files else pd.NA,
    }
])
display(followup_support_summary)


closest_outcome_hba1c = pd.DataFrame(columns=[
    'event_id', 'patient_id', 'drug_class', 'index_date', 'baseline_hba1c_date', 'baseline_hba1c',
    'date', 'code_system', 'code', 'lab_result_num_val', 'units_of_measure', 'days_after_index', 'distance_to_12m'
])

event_base = outcome_event_cohort[['event_id', 'patient_id', 'drug_class', 'index_date', 'baseline_hba1c_date', 'baseline_hba1c']].copy() if not outcome_event_cohort.empty else pd.DataFrame()
if not event_base.empty:
    event_base['patient_id'] = event_base['patient_id'].astype('string')

for path in followup_hba1c_files:
    if not path.exists() or path.stat().st_size == 0:
        continue
    try:
        for chunk in pd.read_csv(path, chunksize=OUTCOME_SELECTION_CHUNKSIZE):
            if event_base.empty:
                continue
            chunk['patient_id'] = chunk['patient_id'].astype('string')
            chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
            chunk['lab_result_num_val'] = pd.to_numeric(chunk['lab_result_num_val'], errors='coerce')
            chunk = chunk[(chunk['units_of_measure'] == '%') & chunk['lab_result_num_val'].notna()].copy()
            if chunk.empty:
                continue

            merged = event_base.merge(chunk, on='patient_id', how='inner')
            if merged.empty:
                continue

            merged['days_after_index'] = (merged['date'] - merged['index_date']).dt.days
            merged = merged[(merged['days_after_index'] >= 90) & (merged['days_after_index'] <= 455)].copy()
            if merged.empty:
                continue
            merged['distance_to_12m'] = (merged['days_after_index'] - 365).abs()
            chunk_best = (
                merged.sort_values(['event_id', 'distance_to_12m', 'date'])
                .groupby('event_id', as_index=False)
                .first()
            )
            if closest_outcome_hba1c.empty:
                closest_outcome_hba1c = chunk_best.copy()
            else:
                combined = pd.concat([closest_outcome_hba1c, chunk_best], ignore_index=True)
                closest_outcome_hba1c = (
                    combined.sort_values(['event_id', 'distance_to_12m', 'date'])
                    .groupby('event_id', as_index=False)
                    .first()
                )
    except pd.errors.EmptyDataError:
        continue

print('Closest outcome HbA1c rows:', len(closest_outcome_hba1c))
print('Closest outcome HbA1c patients:', closest_outcome_hba1c['patient_id'].nunique() if not closest_outcome_hba1c.empty else 0)
if not closest_outcome_hba1c.empty:
    display(closest_outcome_hba1c[['event_id', 'patient_id', 'drug_class', 'index_date', 'date', 'lab_result_num_val', 'days_after_index']].head(10))


,files,size_mb,first_file,last_file
0,19,54.1,batch_001__followup_hba1c.csv,batch_019__followup_hba1c.csv


Closest outcome HbA1c rows: 97673
Closest outcome HbA1c patients: 77016


,event_id,patient_id,drug_class,index_date,date,lab_result_num_val,days_after_index
0,event_0000001,#A##B,GLP1,2019-07-03,2020-07-11,6.7,374
1,event_0000002,#A##B,SGLT2,2023-12-14,2025-02-04,7.0,418
2,event_0000003,#A#0B,GLP1,2023-08-16,2024-06-17,6.4,306
3,event_0000004,#A#1B,SGLT2,2015-07-02,2016-06-21,8.1,355
4,event_0000005,#A#2,DPP4,2018-08-02,2018-11-05,6.1,95
5,event_0000008,#A#9,DPP4,2008-03-10,2009-02-10,6.6,337
6,event_0000009,#A#AB,GLP1,2022-02-09,2023-01-19,6.3,344
7,event_0000011,#A#CB,GLP1,2023-02-07,2024-01-11,7.8,338
8,event_0000012,#A#CB,SGLT2,2024-01-11,2025-03-12,7.8,426
9,event_0000014,#A#DB,DPP4,2016-08-03,2017-08-03,9.0,365


## 8. Apply The Unchanged-Therapy Approximation Using The Full-Project Medication Support Files

This mirrors the validated pilot logic, including the non-background add-on adjustment, but processes the large medication support files in chunks.


In [9]:
MED_CHUNKSIZE = 250_000


def classify_glucose_lowering_class(df):
    cls = pd.Series(pd.NA, index=df.index, dtype='object')
    path = df['path'].fillna('').astype(str) if 'path' in df.columns else pd.Series('', index=df.index)
    desc = df['code_description'].fillna('').astype(str).str.lower() if 'code_description' in df.columns else pd.Series('', index=df.index)

    cls.loc[path.str.contains('/A10BK/', case=False, regex=False) | desc.str.contains('empagliflozin|dapagliflozin|canagliflozin|ertugliflozin')] = 'SGLT2'
    cls.loc[path.str.contains('/A10BH/', case=False, regex=False) | desc.str.contains('sitagliptin|saxagliptin|linagliptin|alogliptin|vildagliptin')] = 'DPP4'
    cls.loc[path.str.contains('/A10BJ/', case=False, regex=False) | desc.str.contains('semaglutide|dulaglutide|liraglutide|exenatide|lixisenatide')] = 'GLP1'
    cls.loc[path.str.contains('/A10BA/', case=False, regex=False) | desc.str.contains('metformin')] = 'METFORMIN'
    cls.loc[path.str.contains('/A10BB/', case=False, regex=False) | desc.str.contains('gliclazide|glipizide|glyburide|glibenclamide|glimepiride')] = 'SULFONYLUREA'
    cls.loc[path.str.contains('/A10BG/', case=False, regex=False) | desc.str.contains('pioglitazone|rosiglitazone')] = 'TZD'
    cls.loc[path.str.contains('/A10A/', case=False, regex=False) | desc.str.contains('insulin')] = 'INSULIN'
    cls.loc[path.str.contains('/A10BF/', case=False, regex=False) | desc.str.contains('acarbose|miglitol')] = 'ALPHA_GLUCOSIDASE'
    other_mask = path.str.contains('/A10BX/', case=False, regex=False) | desc.str.contains('repaglinide|nateglinide|tirzepatide')
    cls.loc[other_mask] = cls.loc[other_mask].fillna('OTHER_A10')
    return cls

outcome_eval = closest_outcome_hba1c.copy()
if not outcome_eval.empty:
    outcome_eval = outcome_eval.rename(columns={'date': 'outcome_hba1c_date', 'lab_result_num_val': 'outcome_hba1c'})

    outcome_events = outcome_eval[['event_id', 'patient_id', 'drug_class', 'index_date', 'outcome_hba1c_date']].copy()
    outcome_events['patient_id'] = outcome_events['patient_id'].astype('string')

    index_continued_events = set()
    background_pairs = set()
    candidate_add_pairs = set()

    glm_files = sorted(FULL_SUPPORT_DIR.glob('batch_*__all_glucose_lowering_meds.csv'))
    print('Medication support files considered:', len(glm_files))

    for path in glm_files:
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            for chunk in pd.read_csv(path, chunksize=MED_CHUNKSIZE):
                if chunk.empty:
                    continue
                chunk['patient_id'] = chunk['patient_id'].astype('string')
                chunk['start_date'] = pd.to_datetime(chunk['start_date'], errors='coerce')
                if 'drug_class' not in chunk.columns or chunk['drug_class'].isna().all():
                    chunk['drug_class'] = classify_glucose_lowering_class(chunk)
                else:
                    chunk['drug_class'] = chunk['drug_class'].fillna(classify_glucose_lowering_class(chunk))
                chunk = chunk.dropna(subset=['patient_id', 'start_date', 'drug_class'])
                if chunk.empty:
                    continue
                meds_class = chunk[['patient_id', 'drug_class', 'start_date']].drop_duplicates()
                merged = outcome_events.merge(meds_class, on='patient_id', how='inner', suffixes=('', '_med'))
                if merged.empty:
                    continue

                idx_cont = merged[
                    (merged['drug_class_med'] == merged['drug_class'])
                    & (merged['start_date'] > merged['index_date'])
                    & (merged['start_date'] <= merged['outcome_hba1c_date'])
                ]
                if not idx_cont.empty:
                    index_continued_events.update(idx_cont['event_id'].astype(str).tolist())

                background = merged[merged['start_date'] < merged['index_date']]
                if not background.empty:
                    background_pairs.update((str(eid), str(cls)) for eid, cls in background[['event_id', 'drug_class_med']].drop_duplicates().itertuples(index=False, name=None))

                add_after = merged[
                    (merged['drug_class_med'] != merged['drug_class'])
                    & (merged['start_date'] > merged['index_date'])
                    & (merged['start_date'] <= merged['outcome_hba1c_date'])
                ]
                if not add_after.empty:
                    candidate_add_pairs.update((str(eid), str(cls)) for eid, cls in add_after[['event_id', 'drug_class_med']].drop_duplicates().itertuples(index=False, name=None))
        except pd.errors.EmptyDataError:
            continue

    non_background_add_pairs = candidate_add_pairs - background_pairs
    new_class_added_events = {event_id for event_id, _ in non_background_add_pairs}

    outcome_eval['index_class_continued'] = outcome_eval['event_id'].astype(str).isin(index_continued_events)
    outcome_eval['new_class_added'] = outcome_eval['event_id'].astype(str).isin(new_class_added_events)
    outcome_eval['unchanged_therapy_approx'] = outcome_eval['index_class_continued'] & (~outcome_eval['new_class_added'])
else:
    outcome_eval = pd.DataFrame(columns=['event_id', 'patient_id', 'drug_class', 'index_date'])

print('Rows with continued index-class prescribing:', int(outcome_eval['index_class_continued'].sum()) if 'index_class_continued' in outcome_eval.columns else 0)
print('Rows with new non-background class added:', int(outcome_eval['new_class_added'].sum()) if 'new_class_added' in outcome_eval.columns else 0)
print('Rows meeting unchanged-therapy approximation:', int(outcome_eval['unchanged_therapy_approx'].sum()) if 'unchanged_therapy_approx' in outcome_eval.columns else 0)
if not outcome_eval.empty:
    display(outcome_eval[['event_id', 'patient_id', 'drug_class', 'index_date', 'outcome_hba1c_date', 'outcome_hba1c', 'index_class_continued', 'new_class_added', 'unchanged_therapy_approx']].head(10))


Medication support files considered: 75


/scratch/local/u1523752/18402540/ipykernel_4020971/471194078.py:39: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=MED_CHUNKSIZE):
/scratch/local/u1523752/18402540/ipykernel_4020971/471194078.py:39: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=MED_CHUNKSIZE):
/scratch/local/u1523752/18402540/ipykernel_4020971/471194078.py:39: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=MED_CHUNKSIZE):
/scratch/local/u1523752/18402540/ipykernel_4020971/471194078.py:39: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=MED_CHUNKSIZE):
/scratch/local/u1523752/18402540/ipykernel_4020971/471194078.py:39: 

Rows with continued index-class prescribing: 68092
Rows with new non-background class added: 25419
Rows meeting unchanged-therapy approximation: 51058


,event_id,patient_id,drug_class,index_date,outcome_hba1c_date,outcome_hba1c,index_class_continued,new_class_added,unchanged_therapy_approx
0,event_0000001,#A##B,GLP1,2019-07-03,2020-07-11,6.7,True,False,True
1,event_0000002,#A##B,SGLT2,2023-12-14,2025-02-04,7.0,False,True,False
2,event_0000003,#A#0B,GLP1,2023-08-16,2024-06-17,6.4,True,False,True
3,event_0000004,#A#1B,SGLT2,2015-07-02,2016-06-21,8.1,False,True,False
4,event_0000005,#A#2,DPP4,2018-08-02,2018-11-05,6.1,False,True,False
5,event_0000008,#A#9,DPP4,2008-03-10,2009-02-10,6.6,True,False,True
6,event_0000009,#A#AB,GLP1,2022-02-09,2023-01-19,6.3,True,False,True
7,event_0000011,#A#CB,GLP1,2023-02-07,2024-01-11,7.8,True,True,False
8,event_0000012,#A#CB,SGLT2,2024-01-11,2025-03-12,7.8,True,True,False
9,event_0000014,#A#DB,DPP4,2016-08-03,2017-08-03,9.0,True,False,True


## 9. Write The Full-Project Outcome Label Files


In [10]:
full_project_outcome_hba1c = outcome_eval.copy()
outcome_eligible_labels = full_project_outcome_hba1c[full_project_outcome_hba1c['unchanged_therapy_approx'] == True].copy() if not full_project_outcome_hba1c.empty else pd.DataFrame()

full_project_outcome_hba1c.to_csv(FULL_PROJECT_OUTCOME_PATH, index=False)
outcome_eligible_labels.to_csv(FULL_PROJECT_OUTCOME_ELIGIBLE_PATH, index=False)

print(FULL_PROJECT_OUTCOME_PATH)
print(FULL_PROJECT_OUTCOME_ELIGIBLE_PATH)
print('Full-project outcome rows:', len(full_project_outcome_hba1c))
print('Full-project outcome patients:', full_project_outcome_hba1c['patient_id'].nunique() if not full_project_outcome_hba1c.empty else 0)
print('Full-project unchanged-therapy outcome rows:', len(outcome_eligible_labels))
print('Full-project unchanged-therapy outcome patients:', outcome_eligible_labels['patient_id'].nunique() if not outcome_eligible_labels.empty else 0)
if not outcome_eligible_labels.empty:
    display(outcome_eligible_labels['drug_class'].value_counts(dropna=False).rename_axis('drug_class').to_frame('rows'))


/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/full_project_outcome_hba1c.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/full_project_outcome_hba1c_eligible.csv
Full-project outcome rows: 97673
Full-project outcome patients: 77016
Full-project unchanged-therapy outcome rows: 51058
Full-project unchanged-therapy outcome patients: 46435


,rows
drug_class,
GLP1,22537
SGLT2,18384
DPP4,10137


## 10. What Notebook 12 Delivers

After running this notebook you will have the full-project outcome label file needed by notebook 11. Once that file exists, rerunning notebook 11 sections 11-13 will make the full-project modeling workflow outcome-ready for the four planned model families.
